# Detector 2D FPN v2 — train trên Kaggle T4

Notebook này train detector 2D với input 640×640 trước khi đưa điểm coarse vào ROI.

Lưu ý: detector_2d_fpn_v2 không phải là tên file. Đây là giá trị ARCHITECTURE bên trong detector_2d.py. Sau khi train, checkpoint sẽ được lưu dưới tên best_detector_2d.pth.

Workflow:

classification has_fire=1/0 + regression p_fire → detector_2d_fpn_v2 → 640×640 → so sánh với best.pth → sau đó mới đưa sang ROI.

Dataset Home Fire YOLO không dùng trong notebook này vì nó có bbox, còn detector này dùng nhãn điểm p_fire.

In [ ]:
from pathlib import Path
import importlib.util
import json
import subprocess
import sys
import time

import torch

print('Python:', sys.version)
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('CUDA version:', torch.version.cuda)
if not torch.cuda.is_available():
    raise RuntimeError('Chưa bật GPU. Vào Settings → Accelerator → GPU T4 rồi Restart Session.')
print('GPU:', torch.cuda.get_device_name(0))
assert 'T4' in torch.cuda.get_device_name(0) or 'Tesla' in torch.cuda.get_device_name(0), torch.cuda.get_device_name(0)

In [ ]:
if importlib.util.find_spec('timm') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'timm'])
import timm
print('timm:', timm.__version__)

In [ ]:
INPUT_ROOT = Path('/kaggle/input')
WORK_ROOT = Path('/kaggle/working')
OUTPUT_DIR = WORK_ROOT / 'detector_2d_v2'

# Để None để tự dò. Chỉ điền đường dẫn khi Kaggle có nhiều bộ input trùng tên.
CODE_ROOT_HINT = None
LABELS_HINT = None
DATASET_ROOT_HINT = None
INIT_CHECKPOINT_HINT = None

IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

def unique_paths(paths):
    result = []
    seen = set()
    for path in paths:
        path = Path(path)
        key = str(path).lower()
        if key not in seen:
            seen.add(key)
            result.append(path)
    return result

def has_cctv_layout(path):
    return any((path / suffix).is_dir() for suffix in ('data/data/img_data', 'data/img_data', 'img_data'))

def find_code_root():
    if CODE_ROOT_HINT is not None:
        candidate = Path(CODE_ROOT_HINT).resolve()
        if (candidate / 'detector_2d.py').is_file() and (candidate / 'train_week6.py').is_file():
            return candidate
    candidates = []
    for root in (INPUT_ROOT, WORK_ROOT):
        if root.is_dir():
            candidates.extend(path.parent for path in root.rglob('detector_2d.py'))
    candidates = [path for path in unique_paths(candidates) if (path / 'train_week6.py').is_file()]
    if not candidates:
        raise FileNotFoundError('Không tìm thấy code root. Hãy upload detector_2d.py, train_week6.py và notebook launcher vào một Kaggle Dataset.')
    candidates.sort(key=lambda path: ('sam' not in str(path).lower(), len(path.parts), str(path).lower()))
    return candidates[0]

def find_labels():
    if LABELS_HINT is not None:
        path = Path(LABELS_HINT).resolve()
        if not path.is_file():
            raise FileNotFoundError(path)
        return path
    candidates = unique_paths(INPUT_ROOT.rglob('dataset_labels (1).json'))
    if not candidates:
        raise FileNotFoundError('Không tìm thấy dataset_labels (1).json dưới /kaggle/input.')
    candidates.sort(key=lambda path: ('fire-model-data' not in str(path).lower(), len(path.parts), str(path).lower()))
    return candidates[0]

def find_dataset_root():
    if DATASET_ROOT_HINT is not None:
        path = Path(DATASET_ROOT_HINT).resolve()
        if not path.is_dir():
            raise FileNotFoundError(path)
        return path
    candidates = []
    for img_root in INPUT_ROOT.rglob('img_data'):
        for ancestor in [img_root] + list(img_root.parents[:4]):
            if ancestor.is_dir() and has_cctv_layout(ancestor):
                candidates.append(ancestor)
    candidates = unique_paths(candidates)
    if not candidates:
        raise FileNotFoundError('Không tìm thấy fire-detection-from-cctv có data/img_data dưới /kaggle/input.')
    candidates.sort(key=lambda path: ('fire-detection-from-cctv' not in str(path).lower(), len(path.parts), str(path).lower()))
    return candidates[0]

def find_checkpoint(name):
    candidates = unique_paths(INPUT_ROOT.rglob(name))
    if not candidates:
        return None
    candidates.sort(key=lambda path: ('week6_spatial' not in str(path).lower(), 'fire-model-data' not in str(path).lower(), len(path.parts), str(path).lower()))
    return candidates[0]

CODE_ROOT = find_code_root()
CODE_SCRIPT = CODE_ROOT / 'kaggle_train_detector_2d.py'
LABELS = find_labels()
DATASET_ROOT = find_dataset_root()
INIT_CHECKPOINT = Path(INIT_CHECKPOINT_HINT).resolve() if INIT_CHECKPOINT_HINT else find_checkpoint('best_spatial.pth')
OLD_CHECKPOINT = find_checkpoint('best.pth')

print('CODE_ROOT =', CODE_ROOT)
print('CODE_SCRIPT =', CODE_SCRIPT)
print('LABELS =', LABELS)
print('DATASET_ROOT =', DATASET_ROOT)
print('INIT_CHECKPOINT =', INIT_CHECKPOINT)
print('OLD_CHECKPOINT =', OLD_CHECKPOINT)
assert CODE_SCRIPT.is_file(), CODE_SCRIPT
assert (CODE_ROOT / 'detector_2d.py').is_file()
assert (CODE_ROOT / 'train_week6.py').is_file()
assert LABELS.is_file()
assert DATASET_ROOT.is_dir()


In [ ]:
def count_images(root):
    return sum(1 for path in root.rglob('*') if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS)

for candidate in (DATASET_ROOT / 'data/data/img_data', DATASET_ROOT / 'data/img_data', DATASET_ROOT / 'img_data'):
    if candidate.is_dir():
        print('IMAGE_ROOT =', candidate)
        for split in ('train', 'val', 'test'):
            split_root = candidate / split
            if split_root.is_dir():
                print(' ', split, '=', count_images(split_root))

sys.path.insert(0, str(CODE_ROOT))
import detector_2d
print('ARCHITECTURE =', detector_2d.ARCHITECTURE)
print('DEFAULT_IMAGE_SIZE =', detector_2d.DEFAULT_IMAGE_SIZE)
assert detector_2d.ARCHITECTURE == 'detector_2d_fpn_v2'
assert detector_2d.DEFAULT_IMAGE_SIZE == (640, 640)
print('Architecture and input size: OK')

In [ ]:
from train_week6 import load_records, split_records
records, load_stats = load_records(LABELS, DATASET_ROOT)
splits = split_records(records, seed=42)
split_info = {name: {'count': len(items), 'fire': sum(int(item.has_fire) for item in items), 'no_fire': sum(int(not item.has_fire) for item in items)} for name, items in splits.items()}
print('load_stats =', load_stats)
print('split_info =', split_info)
if not records or any(value['count'] == 0 for value in split_info.values()):
    raise RuntimeError('Dataset/labels không tạo được đủ train/val/test split.')

## Train

Cấu hình mặc định cho T4: batch 4, gradient accumulation 2, batch hiệu dụng 8, input 640×640, 35 epoch. Nếu thiếu VRAM, đổi batch thành 2 và accumulation thành 4.

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
command = [
    sys.executable, str(CODE_SCRIPT),
    '--input-root', '/kaggle/input',
    '--code-root', str(CODE_ROOT),
    '--labels', str(LABELS),
    '--dataset-root', str(DATASET_ROOT),
    '--output-dir', str(OUTPUT_DIR),
    '--device', '0',
    '--image-size', '640',
    '--batch-size', '4',
    '--grad-accumulation', '2',
    '--epochs', '35',
    '--freeze-epochs', '3',
    '--workers', '2',
]
if INIT_CHECKPOINT is not None and INIT_CHECKPOINT.is_file():
    command.extend(['--init-checkpoint', str(INIT_CHECKPOINT)])
print('Running:')
print(' '.join(str(item) for item in command))
subprocess.run(command, check=True)


In [ ]:
metrics_path = OUTPUT_DIR / 'test_metrics.json'
history_path = OUTPUT_DIR / 'history.json'
best_path = OUTPUT_DIR / 'best_detector_2d.pth'
for path in (metrics_path, history_path, best_path):
    print(path, path.exists(), path.stat().st_size if path.exists() else None)
if not metrics_path.is_file() or not best_path.is_file():
    raise FileNotFoundError('Train chưa tạo đủ best_detector_2d.pth và test_metrics.json')
metrics = json.loads(metrics_path.read_text(encoding='utf-8'))
print(json.dumps(metrics, indent=2, ensure_ascii=False))

In [ ]:
import matplotlib.pyplot as plt
history = json.loads(history_path.read_text(encoding='utf-8'))
epochs = [row['epoch'] for row in history]
train_loss = [row['train']['total'] for row in history]
val_loss = [row['val']['total'] for row in history]
val_f1 = [row['val_metric']['f1'] for row in history]
val_mae = [row['val_metric']['mae_px'] for row in history]
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
axes[0].plot(epochs, train_loss, label='train')
axes[0].plot(epochs, val_loss, label='val')
axes[0].set_title('Total loss')
axes[0].legend()
axes[0].grid()
axes[1].plot(epochs, val_f1)
axes[1].set_title('Validation F1')
axes[1].grid()
axes[2].plot(epochs, val_mae)
axes[2].set_title('Validation MAE pixel')
axes[2].grid()
plt.show()

## So sánh với detector cũ

Phần này chỉ chạy sau khi train xong. Nếu có best.pth trong Kaggle Input, notebook sẽ đánh giá old/new trên cùng split test và cùng threshold.

In [ ]:
COMPARE_SCRIPT = CODE_ROOT / 'compare_detector_2d.py'
COMPARE_OUTPUT = WORK_ROOT / 'detector_2d_comparison'
if OLD_CHECKPOINT is None or not OLD_CHECKPOINT.is_file():
    print('Bỏ qua comparison: không tìm thấy best.pth trong Kaggle Input.')
elif not COMPARE_SCRIPT.is_file():
    print('Bỏ qua comparison: không tìm thấy compare_detector_2d.py.')
else:
    compare_command = [
        sys.executable, str(COMPARE_SCRIPT),
        '--labels', str(LABELS),
        '--dataset', str(DATASET_ROOT),
        '--old', str(OLD_CHECKPOINT),
        '--new', str(best_path),
        '--split', 'test',
        '--max-images', '0',
        '--threshold', '0.5',
        '--device', 'cuda:0',
        '--output-dir', str(COMPARE_OUTPUT),
    ]
    print(' '.join(str(item) for item in compare_command))
    subprocess.run(compare_command, check=True)
    summary_path = COMPARE_OUTPUT / 'summary.json'
    if summary_path.is_file():
        summary = json.loads(summary_path.read_text(encoding='utf-8'))
        for branch, values in summary.get('branches', {}).items():
            print(branch, {key: values.get(key) for key in ('precision', 'recall', 'f1', 'mae_px', 'median_px', 'p95_px', 'pck10', 'pck25', 'latency_ms_mean')})

In [ ]:
from IPython.display import display, FileLink, Image as IPImage
preview = COMPARE_OUTPUT / 'detector_2d_contact_sheet.png'
if preview.is_file():
    display(IPImage(filename=str(preview)))
else:
    print('Chưa có contact sheet comparison.')

archive_path = Path(shutil.make_archive(str(WORK_ROOT / 'detector_2d_v2', 'zip'), 'zip', str(OUTPUT_DIR)))
print('Download:', archive_path)
display(FileLink(str(archive_path)))
if COMPARE_OUTPUT.is_dir():
    comparison_archive = Path(shutil.make_archive(str(WORK_ROOT / 'detector_2d_comparison', 'zip'), 'zip', str(COMPARE_OUTPUT)))
    display(FileLink(str(comparison_archive)))